In [39]:
# ==============================
# 1. CREATE REALISTIC DATASET
# ==============================
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

data = pd.DataFrame({
    # Numerical Features
    "StudyHours": np.random.randint(1, 10, n),
    "SleepHours": np.random.randint(4, 10, n),
    "Attendance": np.random.randint(50, 100, n),
    "PreviousMarks": np.random.randint(40, 100, n),
    "InternetUsage": np.random.randint(1, 8, n),

    # Categorical Features
    "Gender": np.random.choice(["Male", "Female"], n),
    "ParentalEducation": np.random.choice(["HighSchool", "Graduate", "PostGraduate"], n),
    "Extracurricular": np.random.choice(["Yes", "No"], n)
})

# ==============================
# ADD TARGET
# ==============================
score = (
    data["StudyHours"] * 0.3 +
    data["Attendance"] * 0.2 +
    data["PreviousMarks"] * 0.3 -
    data["InternetUsage"] * 0.2
)

data["Pass"] = (score > 50).astype(int)

# ==============================
# INTRODUCE MISSING VALUES (NaN)
# ==============================
for col in data.columns:
    data.loc[data.sample(frac=0.1).index, col] = np.nan  # 10% missing

# Save dataset
data.to_csv("student_mixed_dataset.csv", index=False)

print(data.head())

   StudyHours  SleepHours  Attendance  PreviousMarks  InternetUsage  Gender  \
0         7.0         6.0        75.0           48.0            1.0  Female   
1         4.0         8.0         NaN           59.0            NaN    Male   
2         8.0         9.0        79.0           99.0            5.0  Female   
3         5.0         4.0        60.0           40.0            6.0     NaN   
4         7.0         8.0         NaN           47.0            7.0    Male   

  ParentalEducation Extracurricular  Pass  
0      PostGraduate              No   NaN  
1      PostGraduate              No   0.0  
2      PostGraduate             NaN   0.0  
3          Graduate             Yes   NaN  
4        HighSchool              No   0.0  


In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset
data = pd.read_csv("student_mixed_dataset.csv")


In [41]:
# Remove rows where target is missing
data = data.dropna(subset=["Pass"])

X = data.drop("Pass", axis=1)
y = data["Pass"]

In [42]:
# Identify column types
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns
print("Numerical:", num_cols)
print("Categorical:", cat_cols)

Numerical: Index(['StudyHours', 'SleepHours', 'Attendance', 'PreviousMarks',
       'InternetUsage'],
      dtype='object')
Categorical: Index(['Gender', 'ParentalEducation', 'Extracurricular'], dtype='object')


## NUMERICAL PIPELINE

In [43]:

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# CATEGORICAL PIPELINE

In [44]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


# COMBINE PIPELINES

In [45]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])


In [46]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100))
])


In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [48]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['StudyHours', 'SleepHours', 'Attendance', 'PreviousMarks',
       'InternetUsage'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['Gender', 'ParentalEducation', 'Extracurricular'], dtype='object'))])),
                ('model', RandomForestClassifier())])

In [49]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 1.0


In [50]:
new_data = pd.DataFrame({
    "StudyHours": [6],
    "SleepHours": [7],
    "Attendance": [85],
    "PreviousMarks": [78],
    "InternetUsage": [3],
    "Gender": ["Male"],
    "ParentalEducation": ["Graduate"],
    "Extracurricular": ["Yes"]
})

print("Prediction:", pipeline.predict(new_data))

Prediction: [0.]


In [51]:
pred = pipeline.predict(new_data)[0]

print("Prediction:", "Pass" if int(pred) == 1 else "Fail")

Prediction: Fail


In [52]:
proba = pipeline.predict_proba(new_data)

print("Fail Probability:", proba[0][0])
print("Pass Probability:", proba[0][1])

Fail Probability: 1.0
Pass Probability: 0.0
